# Generator Functions for $\phi$-Divergences

A $\phi$-divergence compares two absolutely continuous measures through the density ratio
$$
s(x)=\frac{d\alpha}{d\beta}(x),\qquad
D_\phi(\alpha|\beta)=\int \phi(s(x))\,d\beta(x).
$$
For discrete measures on a common support this becomes
$$
D_\phi(a|b)=\sum_i b_i\,\phi\!\left(\frac{a_i}{b_i}\right).
$$
This notebook exports two panels: the normalized generator curves and a tiny discrete example showing the ratios and local penalty contributions.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    ORANGE,
    GRAY,
    LIGHT_GRAY,
    DIRAC_MARKER_SIZE,
    box_axes,
    figure_dir,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()
NAME = "dualnorms-phi-generators"
OUT = figure_dir(NAME)


## Generator curves

Affine terms do not change the induced divergence on probability densities.  We use standard representatives normalized by $\phi(1)=0$.

In [ ]:
s = np.linspace(0.035, 3.4, 700)
kl = s * np.log(s) - s + 1
rkl = -np.log(s) + s - 1
tv = np.abs(s - 1)
chi2 = 0.5 * (s - 1) ** 2
hell = (np.sqrt(s) - 1) ** 2
curves = [
    (kl, RED, r"KL"),
    (rkl, BLUE, r"reverse KL"),
    (tv, GRAY, r"TV"),
    (chi2, ORANGE, r"$\chi^2$"),
    (hell, VIOLET, r"Hellinger"),
]

fig, ax = plt.subplots(figsize=(3.0, 2.05))
for vals, color, label in curves:
    ax.plot(s, vals, color=color, lw=1.2, label=label)
ax.scatter([1], [0], s=DIRAC_MARKER_SIZE * 0.9, marker='o', color='black', edgecolor='none', linewidth=0, zorder=4)
ax.axvline(1, color=LIGHT_GRAY, lw=0.7, zorder=0)
ax.axhline(0, color=LIGHT_GRAY, lw=0.7, zorder=0)
ax.set_xlim(0, 3.4)
ax.set_ylim(-0.03, 2.4)
ax.set_xlabel(r"density ratio $s$")
ax.set_ylabel(r"$\phi(s)$")
ax.legend(frameon=False, fontsize=7, loc='upper left', handlelength=1.5)
box_axes(ax)
fig.tight_layout(pad=0.2)
save_pdf(fig, OUT / 'generators.pdf', pad_inches=0.055)
plt.close(fig)


## Discrete ratio penalties

The second panel uses two probability vectors on the same support.  Blue hollow circles encode the reference masses $b_i$, red filled circles encode $a_i$, and the violet curve shows the ratios $s_i=a_i/b_i$.  Orange lollipops show the local KL contributions $b_i\phi(s_i)$.

In [ ]:
x = np.arange(9)
b = np.array([0.07, 0.12, 0.17, 0.15, 0.10, 0.08, 0.11, 0.13, 0.07])
a = np.array([0.04, 0.06, 0.12, 0.22, 0.18, 0.11, 0.07, 0.08, 0.04])
a = a / a.sum()
b = b / b.sum()
ratio = a / b
local_kl = b * (ratio * np.log(ratio) - ratio + 1)

fig, ax = plt.subplots(figsize=(3.1, 2.05))
ax.axhline(1, color=LIGHT_GRAY, lw=0.8, zorder=0)
ax.plot(x, ratio, color=VIOLET, lw=1.1, alpha=0.9, zorder=2)
ax.scatter(x, ratio, s=DIRAC_MARKER_SIZE * 0.85, marker='o', color=VIOLET, edgecolor='none', zorder=3)

mass_scale = 900
ax.scatter(x - 0.13, np.full_like(x, 0.18, dtype=float), s=mass_scale * b, marker='o', facecolors='none', edgecolors=BLUE, linewidths=0.9, zorder=4)
ax.scatter(x + 0.13, np.full_like(x, 0.18, dtype=float), s=mass_scale * a, marker='o', color=RED, alpha=0.72, edgecolor='none', zorder=5)

penalty_scale = 16.0
penalty_height = penalty_scale * local_kl
for xi, yi in zip(x, penalty_height):
    ax.plot([xi, xi], [-0.32, -0.32 + yi], color=ORANGE, lw=1.0, alpha=0.85, zorder=1)
ax.scatter(x, -0.32 + penalty_height, s=DIRAC_MARKER_SIZE * (0.55 + 6.0 * local_kl / local_kl.max()), marker='o', color=ORANGE, edgecolor='none', zorder=4)

ax.text(0.05, 0.93, r"$s_i=a_i/b_i$", transform=ax.transAxes, color=VIOLET, fontsize=8)
ax.text(0.05, 0.11, r"$b_i\phi(s_i)$", transform=ax.transAxes, color=ORANGE, fontsize=8)
ax.set_xlim(-0.6, len(x) - 0.4)
ax.set_ylim(-0.45, max(2.05, ratio.max() + 0.18))
ax.set_xlabel(r"support index $i$")
ax.set_ylabel(r"ratio / contribution")
ax.set_xticks([0, 2, 4, 6, 8])
ax.set_yticks([0, 1, 2])
box_axes(ax)
fig.tight_layout(pad=0.2)
save_pdf(fig, OUT / 'ratio-penalties.pdf', pad_inches=0.055)
plt.close(fig)
